<a href="https://colab.research.google.com/github/Coder2264/BTP/blob/main/Implementation2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch-geometric==2.7.0  # Latest version
!pip install transformers  # For modern tokenization and potential embeddings
!pip install numpy pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch_geometric.nn as pyg_nn
from torch_geometric.data import Data
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
from transformers import AutoTokenizer
from collections import Counter
import itertools

In [4]:
df = pd.read_csv('/content/processed2_dataset.csv')
texts = df['input_text'].tolist()
labels = df.iloc[:, 1:].values  # Numpy array of shape (26000, 20)

In [5]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
max_len = 128  # Adjust based on your data (analyze avg text length; modern default for efficiency)
tokenized_texts = tokenizer(texts, padding='max_length', truncation=True, max_length=max_len, return_tensors='pt')
input_ids = tokenized_texts['input_ids']  # Shape (num_samples, max_len)
attention_mask = tokenized_texts['attention_mask']  # For masking padding

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
X_train_ids, X_temp_ids, y_train, y_temp = train_test_split(input_ids.numpy(), labels, test_size=0.2, random_state=42)
X_train_mask, X_temp_mask, _, _ = train_test_split(attention_mask.numpy(), labels, test_size=0.2, random_state=42)
X_val_ids, X_test_ids, y_val, y_test = train_test_split(X_temp_ids, y_temp, test_size=0.5, random_state=42)
X_val_mask, X_test_mask, _, _ = train_test_split(X_temp_mask, y_temp, test_size=0.5, random_state=42)

# Convert back to tensors
X_train_ids = torch.tensor(X_train_ids, dtype=torch.long)
X_val_ids = torch.tensor(X_val_ids, dtype=torch.long)
X_test_ids = torch.tensor(X_test_ids, dtype=torch.long)
X_train_mask = torch.tensor(X_train_mask, dtype=torch.long)
X_val_mask = torch.tensor(X_val_mask, dtype=torch.long)
X_test_mask = torch.tensor(X_test_mask, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [7]:
class TextDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.attention_masks[idx], self.labels[idx]

train_dataset = TextDataset(X_train_ids, X_train_mask, y_train)
val_dataset = TextDataset(X_val_ids, X_val_mask, y_val)
test_dataset = TextDataset(X_test_ids, X_test_mask, y_test)

batch_size = 32  # Adjust based on GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [8]:
num_labels = 20
label_cooccur = np.dot(y_train.T, y_train) > 0  # Binary: edge if co-occur at least once
label_cooccur = label_cooccur.astype(np.float32)
label_cooccur += np.eye(num_labels)  # Add self-loops

# Normalize for GCN (optional, but best practice)
degrees = np.sum(label_cooccur, axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(degrees + 1e-10))  # Avoid division by zero
adj_norm = D_inv_sqrt @ label_cooccur @ D_inv_sqrt
adj_norm = torch.tensor(adj_norm, dtype=torch.float32)

# For torch-geometric: edge_index from non-zero elements
edge_index = torch.nonzero(torch.tensor(label_cooccur), as_tuple=False).t().contiguous()

/tmp/ipython-input-4163064041.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  label_cooccur = np.dot(y_train.T, y_train) > 0  # Binary: edge if co-occur at least once


In [9]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)  # Learnable; can init with pre-trained if desired
        self.bilstm = nn.LSTM(embed_dim, hidden_dim, num_layers, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)  # Reduce dim

    def forward(self, input_ids, attention_mask):
        embed = self.embedding(input_ids)
        # Pack sequence to handle variable lengths (modern best practice)
        lengths = attention_mask.sum(dim=1).cpu()
        packed_embed = nn.utils.rnn.pack_padded_sequence(embed, lengths, batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.bilstm(packed_embed)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(hidden)  # (batch, hidden_dim)

class GNNWithAttention(nn.Module):
    def __init__(self, num_labels, label_embed_dim, hidden_dim, num_heads=4):  # Increase heads for better capture
        super().__init__()
        self.label_embeds = nn.Parameter(torch.randn(num_labels, label_embed_dim))
        self.gat = pyg_nn.GATv2Conv(label_embed_dim, hidden_dim, heads=num_heads, concat=False)  # Latest GAT, average heads

    def forward(self, edge_index):
        h = self.gat(self.label_embeds, edge_index)
        return h  # (num_labels, hidden_dim)

class OverallModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_labels, label_embed_dim, num_heads):
        super().__init__()
        self.bilstm = BiLSTM(vocab_size, embed_dim, hidden_dim, num_layers)
        self.gnn = GNNWithAttention(num_labels, label_embed_dim, hidden_dim, num_heads)

    def forward(self, input_ids, attention_mask, edge_index):
        text_feat = self.bilstm(input_ids, attention_mask)
        label_feat = self.gnn(edge_index)
        logits = torch.matmul(text_feat, label_feat.T)  # (batch, num_labels)
        return logits  # Return logits for BCEWithLogitsLoss

# Hyperparams (tuned for modernity; adjust)
vocab_size = tokenizer.vocab_size  # From HF tokenizer
embed_dim = 128  # Smaller for efficiency
hidden_dim = 256
num_layers = 2  # Deeper for better features
label_embed_dim = 128
num_heads = 4

model = OverallModel(vocab_size, embed_dim, hidden_dim, num_layers, num_labels, label_embed_dim, num_heads)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
edge_index = edge_index.to(device)

In [10]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

epochs = 20  # More epochs with scheduler
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for input_ids, attn_mask, labels in train_loader:
        input_ids, attn_mask, labels = input_ids.to(device), attn_mask.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(input_ids, attn_mask, edge_index)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}')

    # Validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for input_ids, attn_mask, labels in val_loader:
            input_ids, attn_mask, labels = input_ids.to(device), attn_mask.to(device), labels.to(device)
            logits = model(input_ids, attn_mask, edge_index)
            val_loss += criterion(logits, labels).item()
    print(f'Val Loss: {val_loss / len(val_loader)}')
    scheduler.step(val_loss / len(val_loader))

Epoch 1, Loss: 0.3045336594255708
Val Loss: nan
Epoch 2, Loss: 0.22496391489780207
Val Loss: nan
Epoch 3, Loss: 0.19215896033125815
Val Loss: nan
Epoch 4, Loss: 0.15113225983415576
Val Loss: nan
Epoch 5, Loss: 0.13955965215782468
Val Loss: nan
Epoch 6, Loss: 0.13071802189881854
Val Loss: nan
Epoch 7, Loss: 0.1197384523938028
Val Loss: nan
Epoch 8, Loss: 0.11798991464453636
Val Loss: nan
Epoch 9, Loss: 0.11672762736356515
Val Loss: nan
Epoch 10, Loss: 0.11535247626493303
Val Loss: nan
Epoch 11, Loss: 0.115218261050449
Val Loss: nan
Epoch 12, Loss: 0.11506293837329466
Val Loss: nan
Epoch 13, Loss: 0.1149114655076171
Val Loss: nan
Epoch 14, Loss: 0.11489193162900939
Val Loss: nan
Epoch 15, Loss: 0.11488086808499673
Val Loss: nan
Epoch 16, Loss: 0.11487824245751332
Val Loss: nan
Epoch 17, Loss: 0.11487349067017329
Val Loss: nan
Epoch 18, Loss: 0.11486774377471251
Val Loss: nan
Epoch 19, Loss: 0.11486163718237294
Val Loss: nan
Epoch 20, Loss: 0.11487487967816189
Val Loss: nan


In [11]:
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for input_ids, attn_mask, labels in test_loader:
        input_ids, attn_mask = input_ids.to(device), attn_mask.to(device)
        logits = model(input_ids, attn_mask, edge_index)
        preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)
micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(all_labels.ravel(), all_preds.ravel(), average='micro', zero_division=0)
print(f'Micro Precision: {micro_p}, Recall: {micro_r}, F1: {micro_f1}')

Micro Precision: 0.925953237410072, Recall: 0.925953237410072, F1: 0.925953237410072
